# Experiment 2 — KFold CV, Per-Item Imputation, GBM Ensemble

**Changes from Experiment 1:**

| Change | Reason |
|---|---|
| Single split → 5-fold KFold CV | More stable evaluation; not sensitive to one random draw |
| Global mean weight → per-item mean | Weight is a product property — if OUT013 has a weight for FDA15 and OUT049 doesn't, fill from OUT013 |
| Global mode outlet size → mode per outlet type | Grocery Stores are always Small; this reflects the actual pattern |
| Visibility zeros → per-item mean | Zero display area is physically impossible — treat as missing |
| `Outlet_Age = 2026 - year` → `2013 - year` | Dataset is from 2013 |
| RF only → RF + GBM ensemble | More model diversity |
| Added `Is_Grocery`, `ItemType_OutletType` | Additional categorical signals |

**Notes:**
- KFold OOF is much more stable than a single split
- Per-item imputation is clearly the right approach for weight
- GBM added marginal OOF improvement over RF alone — but as we discovered later, GBDTs actually hurt on this dataset's leaderboard

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ── 1. Load ───────────────────────────────────────────────────────────────────
train = pd.read_csv('train_v9rqX0R.csv')
test  = pd.read_csv('test_AbJTz2l.csv')

# ── 2. Cleaning — train stats only applied to test (no leakage) ──────────────
def clean(df, ref_weight=None, ref_vis=None):
    df = df.copy()
    fat_map = {'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'}
    df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_map)
    df['Item_Category'] = df['Item_Identifier'].str[:2]
    df.loc[df['Item_Category'] == 'NC', 'Item_Fat_Content'] = 'Non-Consumable'

    # Item_Weight: fill using per-item means from reference set (train)
    if ref_weight is None:
        ref_weight = df.groupby('Item_Identifier')['Item_Weight'].mean()
    df['Item_Weight'] = df.apply(
        lambda r: ref_weight.get(r['Item_Identifier'], np.nan)
        if pd.isna(r['Item_Weight']) else r['Item_Weight'], axis=1)
    df['Item_Weight'].fillna(df['Item_Weight'].median(), inplace=True)

    df['Outlet_Size'] = df.groupby('Outlet_Type')['Outlet_Size'].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Medium'))

    # Item_Visibility: zero → per-item mean from reference set
    df['Item_Visibility'] = df['Item_Visibility'].replace(0, np.nan)
    if ref_vis is None:
        ref_vis = df.groupby('Item_Identifier')['Item_Visibility'].mean()
    df['Item_Visibility'] = df.apply(
        lambda r: ref_vis.get(r['Item_Identifier'], np.nan)
        if pd.isna(r['Item_Visibility']) else r['Item_Visibility'], axis=1)
    df['Item_Visibility'].fillna(df['Item_Visibility'].mean(), inplace=True)

    df['Outlet_Age']  = 2013 - df['Outlet_Establishment_Year']
    df['Is_Grocery']  = (df['Outlet_Type'] == 'Grocery Store').astype(int)
    df['MRP_Cluster'] = pd.cut(df['Item_MRP'], bins=[0,70,130,200,270],
                                labels=[0,1,2,3]).astype(int)
    df['ItemType_OutletType'] = (df['Item_Type'].astype(str) + '_' +
                                  df['Outlet_Type'].astype(str))
    return df, ref_weight, ref_vis

train_c, ref_w, ref_v = clean(train)
test_c,  _,     _     = clean(test, ref_weight=ref_w, ref_vis=ref_v)

# Outlet_MRP_Mean: computed from train only
outlet_mrp = train_c.groupby('Outlet_Identifier')['Item_MRP'].mean()
train_c['Outlet_MRP_Mean'] = train_c['Outlet_Identifier'].map(outlet_mrp)
test_c['Outlet_MRP_Mean']  = test_c['Outlet_Identifier'].map(outlet_mrp)

# Visibility ratio: item means from train only
vis_mean = train_c.groupby('Item_Identifier')['Item_Visibility'].mean()
train_c['Visibility_MeanRatio'] = train_c['Item_Visibility'] / (train_c['Item_Identifier'].map(vis_mean) + 1e-6)
test_c['Visibility_MeanRatio']  = test_c['Item_Visibility']  / (test_c['Item_Identifier'].map(vis_mean).fillna(vis_mean.mean()) + 1e-6)

# ── 3. Encode — fit on combined to handle all categories ─────────────────────
cat_cols = ['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size',
            'Outlet_Location_Type', 'Outlet_Type', 'Item_Category', 'ItemType_OutletType']
for col in cat_cols:
    le = LabelEncoder().fit(pd.concat([train_c[col], test_c[col]]).astype(str))
    train_c[col] = le.transform(train_c[col].astype(str))
    test_c[col]  = le.transform(test_c[col].astype(str))

# ── 4. Feature matrix ─────────────────────────────────────────────────────────
feats = ['Item_MRP', 'Outlet_Type', 'Outlet_Identifier', 'Outlet_Age',
         'Item_Visibility', 'Item_Weight', 'Item_Type', 'Item_Category',
         'Is_Grocery', 'MRP_Cluster', 'Outlet_Size', 'Outlet_Location_Type',
         'Visibility_MeanRatio', 'Outlet_MRP_Mean', 'Item_Fat_Content',
         'ItemType_OutletType']

for df in [train_c, test_c]:
    for f in feats:
        df[f] = df[f].fillna(df[f].median())

X      = train_c[feats].values.astype(float)
y      = train['Item_Outlet_Sales'].values
X_test = test_c[feats].values.astype(float)

print(f"Train: {X.shape}  |  Test: {X_test.shape}")

# ── 5. Models ─────────────────────────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

models = {
    'RF': RandomForestRegressor(
        n_estimators=1000, min_samples_leaf=20,
        max_features=0.5, n_jobs=-1, random_state=SEED
    ),
    'GBR': GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        min_samples_leaf=10, subsample=0.8, max_features=0.8,
        random_state=SEED
    ),
}

oof_all  = {}
test_all = {}

for name, mdl in models.items():
    oof    = np.zeros(len(X))
    t_pred = np.zeros(len(X_test))
    for tr_i, val_i in kf.split(X):
        mdl.fit(X[tr_i], y[tr_i])
        oof[val_i] = mdl.predict(X[val_i])
        t_pred    += mdl.predict(X_test) / 5
    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"{name:6s}  OOF RMSE: {rmse:.2f}")
    oof_all[name]  = oof
    test_all[name] = t_pred

# ── 6. Ensemble ───────────────────────────────────────────────────────────────
rmse_scores = {n: np.sqrt(mean_squared_error(y, oof_all[n])) for n in models}
weights = {k: 1/v for k,v in rmse_scores.items()}
total_w = sum(weights.values())
weights = {k: v/total_w for k,v in weights.items()}
print(f"\nWeights: { {k: round(v,3) for k,v in weights.items()} }")

ens_oof  = sum(weights[n] * oof_all[n]  for n in models)
ens_test = sum(weights[n] * test_all[n] for n in models)
print(f"Ensemble OOF RMSE: {np.sqrt(mean_squared_error(y, ens_oof)):.2f}")

# ── 7. Save both RF-only and ensemble predictions ─────────────────────────────
for label, preds in [('rf_only', test_all['RF']), ('ensemble', ens_test)]:
    sub = test[['Item_Identifier', 'Outlet_Identifier']].copy()
    sub['Item_Outlet_Sales'] = preds.clip(min=0)
    path = f'submission/predictions2_{label}.csv'
    sub.to_csv(path, index=False)
    print(f"Saved {path}")
    print(sub['Item_Outlet_Sales'].describe().round(2))
    print()

Train: (8523, 16)  |  Test: (5681, 16)
RF      OOF RMSE: 1083.51
GBR     OOF RMSE: 1119.75

Weights: {'RF': np.float64(0.508), 'GBR': np.float64(0.492)}
Ensemble OOF RMSE: 1092.62
Saved submission/predictions2_rf_only.csv
count    5681.00
mean     2180.92
std      1327.34
min        93.67
25%      1112.68
50%      2074.50
75%      3067.51
max      6299.38
Name: Item_Outlet_Sales, dtype: float64

Saved submission/predictions2_ensemble.csv
count    5681.00
mean     2178.49
std      1337.20
min        34.06
25%      1089.48
50%      2078.49
75%      3066.52
max      6774.97
Name: Item_Outlet_Sales, dtype: float64

